# Buổi 6 — Lab

Chạy từng ô từ trên xuống. Mỗi bước ứng với một mục trong tài liệu (mục 5). Bạn sửa `phan_ra.py`; các ô tự dùng bản mới.

In [ ]:
# sửa tệp .py trong code/ thì các ô sau tự dùng bản mới, không cần khởi động lại
%load_ext autoreload
%autoreload 2

## Bước 1 — Chạy code đầu buổi (chạy lại ô này sau bước 3)

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import phan_ra as pr
from statsmodels.tsa.seasonal import seasonal_decompose

y = pr.doc_nhu_cau()
print(f"{len(y)} giờ, NaN {y.isna().sum()}, {y.index[0]} → {y.index[-1]}")
bang = pr.phan_ra(pr.lap_cho_trong(y))
print({k: round(v, 3) for k, v in pr.do_manh(bang).items()})
for theo in (("dayofweek", "hour"), ("month", "hour")):
    print(theo, "mẫu hình còn trong phần dư:", round(pr.ty_le_mau_hinh_con_lai(bang, theo), 4))

## Bước 2 — Tự viết phân rã cổ điển (mục 4.1), rồi xem mùa vụ tuần trốn ở đâu (mục 4.3)

In [ ]:
def co_dien(x, m):
    x = np.asarray(x, dtype=float)
    w = np.r_[0.5, np.ones(m - 1), 0.5] / m
    T = np.full(x.size, np.nan)
    T[m // 2 : -m // 2] = np.convolve(x, w, mode="valid")
    pha = np.arange(x.size) % m
    S_mua = np.array([np.nanmean((x - T)[pha == k]) for k in range(m)])
    S = (S_mua - S_mua.mean())[pha]
    return T, S, x - T - S


quy = [22, 18, 15, 29, 30, 22, 19, 33, 30, 26, 23, 37]
T, S, R = co_dien(quy, 4)
print(pd.DataFrame({"y": quy, "T": T, "S": S, "R": R}))

In [ ]:
x = pr.lap_cho_trong(y)
T, S, R = co_dien(x, 24)
kq = seasonal_decompose(x, period=24)
print("lệch lớn nhất so với statsmodels:", np.nanmax(np.abs(T - kq.trend.to_numpy())), np.max(np.abs(S - kq.seasonal.to_numpy())))
xu_huong = pd.Series(T, index=x.index.tz_localize("UTC").tz_convert(pr.MUI_GIO))
xu_huong.groupby(xu_huong.index.dayofweek).mean().round(0)   # 0 = thứ Hai

## Bước 3 — Đổi sang MSTL (24, 168)

Sửa `phan_ra` trong `phan_ra.py` (mục 4.4), rồi chạy lại ô bước 1. Chấm: `python lab.py check` trong terminal.

## Bước 4 — Robust quanh giờ hỏng 21/11/2024 (mục 4.6)

In [ ]:
khong, co = pr.phan_ra(x), pr.phan_ra(x, robust=True)
print("phần dư 21/11 17:00 UTC:", round(khong.loc["2024-11-21 17:00", "resid"]), round(co.loc["2024-11-21 17:00", "resid"]))
gio = pd.date_range("2024-11-18 17:00", "2024-11-25 17:00", freq="D")
pd.DataFrame({"không robust": khong.loc[gio, "seasonal_24"], "robust": co.loc[gio, "seasonal_24"]}).round(0)

## Bước 5 — Mùa vụ ngày theo tháng (hình cuối mục 4.4)

In [ ]:
s24 = khong["seasonal_24"]
s24.index = s24.index.tz_localize("UTC").tz_convert(pr.MUI_GIO)
ho_so = s24.groupby([s24.index.month, s24.index.hour]).mean().unstack(0) / 1000
ax = ho_so[[1, 4, 7, 10]].plot(figsize=(8, 3.5), xlabel="giờ New York", ylabel="GW")
ax.set_title("Mùa vụ ngày theo tháng")
print("biên độ (GW):", (ho_so.max() - ho_so.min()).round(1).to_dict())